# Sesión 3. RDDs en PySpark

Creación de RDDs, funciones lambda, transformaciones y acciones.

In [1]:
# Instalar SDK Java (25 si está disponible, si no 21)
!sudo apt-get update -qq > /dev/null
!sudo apt-get install -y openjdk-25-jdk-headless -qq > /dev/null 2>&1 || \
 sudo apt-get install -y openjdk-21-jdk-headless -qq > /dev/null

# Descargar Spark 4.2.0
!wget -q https://archive.apache.org/dist/spark/spark-4.2.0/spark-4.2.0-bin-hadoop3.tgz
# Descomprimir el archivo descargado
!tar xf spark-4.2.0-bin-hadoop3.tgz

# Configurar variables de entorno
import os
os.environ["SPARK_HOME"] = "/content/spark-4.2.0-bin-hadoop3"

# JAVA_HOME apunta al JDK que realmente se haya instalado
for version in (25, 21, 17):
    ruta = f"/usr/lib/jvm/java-{version}-openjdk-amd64"
    if os.path.isdir(ruta):
        os.environ["JAVA_HOME"] = ruta
        break

# Desinstalar dataproc-spark-connect
!pip uninstall -y -q dataproc-spark-connect
# Instalar findspark
!pip install -q findspark
# Instalar pyspark
!pip install -q pyspark==4.2.0

# Se importa la librería findspark
import findspark
findspark.init()

print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.1/450.1 MB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
JAVA_HOME: /usr/lib/jvm/java-25-openjdk-amd64


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("Sesion3_RDDs").getOrCreate()

# sc es el punto de entrada para trabajar con RDDs
sc = spark.sparkContext

In [7]:
# Según chatgpt
# RDD vs DataFrame en PySpark
# La diferencia más importante podemos verlo así:

# Característica RDD
# Nivel:	Bajo
# Estructura:	Objetos
# Schema:	No necesariamente	Sí
# Estilo:	Programación funcional
# Optimización automática:	Limitada
# Facilidad de uso:	Más complejo
# Rendimiento:	Puede ser menor
# Flexibilidad:	Muy alta
# Uso actual:	Casos específicos

# Característica  DataFrame
# Nivel:	Alto
# Estructura:	Filas y columnas
# Schema:	Sí
# Estilo:	SQL/tabular
# Optimización automática:	Mucha
# Facilidad de uso:	Más sencillo
# Rendimiento:	Generalmente mejor
# Flexibilidad:	Alta, pero estructurada
# Uso actual:	La mayoría de trabajos de datos

# Mas adelante veremos que:
#              Spark
#                │
#        ┌───────┴────────┐
#        │                │
#       RDD          DataFrame
#        │                │
# bajo nivel         alto nivel
#        │                │
# map/filter       select/filter/groupBy

# Creando un RDD

Un RDD es una colección de datos repartida entre los nodos del clúster. Es inmutable, así que ninguna operación lo modifica. Todas devuelven un RDD nuevo.

## parallelize

Crea un RDD a partir de una colección de Python, por ejemplo una lista o un rango.

**Tipo**: Creación

## collect

Trae todos los elementos del RDD al nodo controlador y los devuelve como una lista de Python.

**Tipo**: Acción

**Cuidado**: `collect()` carga el RDD completo en la memoria del controlador, así que con datos grandes se queda sin memoria. En producción se usa `take()` o `count()`.

In [3]:
# Ejemplo 1: RDD a partir de una lista
rdd_1 = sc.parallelize([1, 2, 3, 4, 5])

In [4]:
type(rdd_1)

pyspark.core.rdd.RDD

In [5]:
print("RDD a partir de una lista:", rdd_1.collect())

RDD a partir de una lista: [1, 2, 3, 4, 5]


### Los RDD son inmutables

Una transformación nunca modifica el RDD original. Siempre devuelve uno nuevo.

In [6]:
# El RDD original queda igual después de aplicarle map
rdd_mas_uno = rdd_1.map(lambda x: x + 1)

print("Nuevo:   ", rdd_mas_uno.collect())
print("Original:", rdd_1.collect())

Nuevo:    [2, 3, 4, 5, 6]
Original: [1, 2, 3, 4, 5]


Tampoco se puede cambiar un elemento suelto. La siguiente celda falla a propósito y lanza un `TypeError`.

In [17]:
# rdd_1[0] = 6

In [10]:
# Ejemplo 2: RDD de cadenas de texto
rdd_2 = sc.parallelize(["Hola", "Mundo", "PySpark", "RDD"])
print(rdd_2.collect())

['Hola', 'Mundo', 'PySpark', 'RDD']


In [11]:
# Ejemplo 3: RDD con pares clave-valor
rdd_3 = sc.parallelize([("A", 1), ("B", 2), ("C", 3)])
print(rdd_3.collect())

[('A', 1), ('B', 2), ('C', 3)]


In [12]:
# Ejemplo 4: RDD a partir de una secuencia de números
rdd_4 = sc.parallelize(range(1, 11))
print(rdd_4.collect())

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [13]:
# Ejemplo 5: RDD vacío
rdd_5 = sc.emptyRDD()
print(rdd_5.collect())

[]


In [14]:
# Ejemplo 6: RDD vacío a partir de una lista vacía
rdd_6 = sc.parallelize([])
print(rdd_6.collect())

[]


In [15]:
# Ejemplo 7: RDD repartido en 3 particiones
rdd_7 = sc.parallelize([1, 2, 3, 4, 5], numSlices=3)

print("Particiones:", rdd_7.getNumPartitions())
print("Contenido:  ", rdd_7.glom().collect())  # glom() agrupa por partición

Particiones: 3
Contenido:   [[1], [2, 3], [4, 5]]


In [16]:
# Ejemplo 8: RDD a partir de un rango
rdd_rango = sc.range(1, 101, step=5)
print(rdd_rango.collect())

[1, 6, 11, 16, 21, 26, 31, 36, 41, 46, 51, 56, 61, 66, 71, 76, 81, 86, 91, 96]


## textFile

Lee un archivo de texto y crea un RDD con una línea por elemento.

**Tipo**: Creación

Igual que `parallelize`, `textFile` no es una transformación, porque no recibe un RDD sino que lo produce. Eso sí, es perezosa. El archivo no se lee hasta la primera acción.

In [ ]:
def crear_archivo_para_rdd():
    nombre_archivo = "datos_ejemplo.txt"
    with open(nombre_archivo, "w") as f:
        f.write("Línea 1: Esto es un ejemplo de archivo de texto.\n")
        f.write("Línea 2: PySpark es genial para procesamiento distribuido.\n")
        f.write("Línea 3: Aquí hay otra línea de texto.\n")
        f.write("Línea 4: Aprendiendo cómo leer archivos con PySpark.\n")
    print(f"Archivo '{nombre_archivo}' creado.")


crear_archivo_para_rdd()

In [ ]:
rdd_archivo = sc.textFile("datos_ejemplo.txt")

print("\n".join(rdd_archivo.collect()))

## Ejercicios: creación de RDDs

**1.** Crea un RDD con los números enteros del 1 al 10 y muestra sus elementos.

**2.** Genera un archivo `texto_entrada.txt` con una función de Python parecida a `crear_archivo_para_rdd()`, léelo con Spark y muestra su contenido.

**3.** Crea un RDD vacío con 5 particiones.

**4.** Crea un RDD con los números primos entre 1 y 20.

**5.** Crea dos RDD distintos, uno con `["Ana", "Carlos", "Beatriz"]` y otro con `["Lucas", "Maria", "Victor"]`.

In [ ]:
# Problema 1


In [ ]:
# Problema 2


In [ ]:
# Problema 3


In [ ]:
# Problema 4


In [ ]:
# Problema 5


# Funciones lambda

Una lambda es una función sin nombre. Sirve cuando solo la vas a usar en un lugar, y en Spark aparecen todo el tiempo como argumento de `map`, `filter` y compañía.

    lambda argumentos: expresión

In [ ]:
def suma(x, y):
    return x + y


suma(3, 2)

In [ ]:
suma_lambda = lambda x, y: x + y

suma_lambda(3, 2)

In [ ]:
# Las dos formas dan el mismo resultado
assert suma(3, 2) == suma_lambda(3, 2)

In [ ]:
duplica = lambda x: x * 2

print(duplica(4))

In [ ]:
concatenar = lambda a, b: a + b

print(concatenar("Hola, ", "Mundo"))

# Transformaciones

Una transformación devuelve un RDD nuevo y no ejecuta nada. Spark solo la anota y espera. El cálculo ocurre hasta que llamas una acción.

## map

Aplica una función a cada elemento. El RDD que sale tiene el mismo número de elementos que el original.

**Tipo**: Transformación

In [ ]:
# Ejemplo 1: Incrementar cada número en 1 usando una función con nombre
def incrementar_en_uno(numero):
    return numero + 1


rdd = sc.parallelize([1, 2, 3, 4, 5])
print(rdd.map(incrementar_en_uno).collect())

In [ ]:
# Ejemplo 2: Lo mismo, con una lambda
rdd = sc.parallelize([1, 2, 3, 4, 5])
print(rdd.map(lambda x: x + 1).collect())

In [ ]:
# Ejemplo 3: Convertir cada cadena a mayúsculas
rdd_texto = sc.parallelize(["hola", "mundo"])
print(rdd_texto.map(lambda x: x.upper()).collect())

In [ ]:
# Ejemplo 4: Sumar los valores de cada fila
rdd_filas = sc.parallelize([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(rdd_filas.map(sum).collect())

### Evaluación perezosa

La siguiente celda define una operación imposible, sumarle un número a un texto, y aun así no falla. La transformación todavía no se ejecuta.

In [ ]:
rdd_texto = sc.parallelize(["hola", "adios"])
rdd_error = rdd_texto.map(lambda x: x + 1)

El error aparece hasta que pedimos el resultado con una acción.

In [ ]:
print(rdd_error.collect())

In [ ]:
# Si concatenamos un texto en lugar de un número, sí funciona
print(rdd_texto.map(lambda x: x + "1").collect())

## flatMap

Como `map`, pero cada elemento puede producir cero, uno o varios resultados, y la salida se aplana en un solo RDD.

**Tipo**: Transformación

In [ ]:
# Ejemplo 1: Separar cada frase en palabras
rdd_frases = sc.parallelize(["hola mundo", "aprendiendo PySpark"])
print(rdd_frases.flatMap(lambda x: x.split(" ")).collect())

In [ ]:
# Con map cada frase queda en una lista aparte
print(rdd_frases.map(lambda x: x.split(" ")).collect())

In [ ]:
# Ejemplo 2: Generar un rango a partir de cada número
rdd_numeros = sc.parallelize([2, 3, 5])
print(rdd_numeros.flatMap(lambda x: range(1, x)).collect())

In [ ]:
# Ejemplo 3: Aplanar una lista de listas
rdd_anidado = sc.parallelize([[1, 2], [3, 4, 5]])
print(rdd_anidado.flatMap(lambda x: x).collect())

In [ ]:
# Ejemplo 4: Repetir cada palabra dos veces
rdd_palabras = sc.parallelize(["sol", "luna"])
print(rdd_palabras.flatMap(lambda x: [x, x]).collect())

## filter

Se queda con los elementos que cumplen una condición.

**Tipo**: Transformación

In [ ]:
# Ejemplo 1: Quedarse solo con los números pares
rdd = sc.parallelize([1, 2, 3, 4, 5, 6])
print(rdd.filter(lambda x: x % 2 == 0).collect())

In [ ]:
# Ejemplo 2: Quedarse con las palabras que contienen una 'b'
rdd_texto = sc.parallelize(["apple", "banana", "pear"])
print(rdd_texto.filter(lambda x: "b" in x).collect())

In [ ]:
# Ejemplo 3: Quedarse con los números mayores que 3
rdd = sc.parallelize([1, 2, 3, 4, 5])
print(rdd.filter(lambda x: x > 3).collect())

## distinct

Elimina los elementos repetidos.

**Tipo**: Transformación

In [ ]:
# Ejemplo 1: Valores únicos de un RDD de números
rdd = sc.parallelize([1, 2, 2, 3, 4, 4, 5])
print(sorted(rdd.distinct().collect()))

In [ ]:
# Ejemplo 2: Valores únicos de un RDD de cadenas
rdd_texto = sc.parallelize(["a", "b", "a", "c", "b"])
print(sorted(rdd_texto.distinct().collect()))

In [ ]:
# Ejemplo 3: Valores únicos de un RDD de tuplas
rdd_pares = sc.parallelize([("a", 1), ("b", 2), ("a", 1)])
print(sorted(rdd_pares.distinct().collect()))

## union

Junta dos RDD en uno solo.

**Tipo**: Transformación

In [ ]:
# Ejemplo 1: Unir dos RDD de números
rdd_a = sc.parallelize([1, 2, 3])
rdd_b = sc.parallelize([4, 5, 6])
print(rdd_a.union(rdd_b).collect())

In [ ]:
# Ejemplo 2: union no elimina duplicados
rdd_a = sc.parallelize([1, 2, 3])
rdd_b = sc.parallelize([3, 4, 5])
print(rdd_a.union(rdd_b).collect())

In [ ]:
# Ejemplo 3: Unir dos RDD de cadenas
rdd_a = sc.parallelize(["a", "b"])
rdd_b = sc.parallelize(["c", "d"])
print(rdd_a.union(rdd_b).collect())

## intersection

Devuelve los elementos que están en los dos RDD.

**Tipo**: Transformación

In [ ]:
# Ejemplo 1: Elementos comunes a dos RDD
rdd_a = sc.parallelize([1, 2, 3, 4])
rdd_b = sc.parallelize([3, 4, 5, 6])
print(sorted(rdd_a.intersection(rdd_b).collect()))

In [ ]:
# Ejemplo 2: Si no hay elementos en común, el resultado es vacío
rdd_a = sc.parallelize([1, 2, 3])
rdd_b = sc.parallelize([4, 5, 6])
print(rdd_a.intersection(rdd_b).collect())

In [ ]:
# Ejemplo 3: Intersección de dos RDD de palabras
rdd_a = sc.parallelize(["hola", "mundo"])
rdd_b = sc.parallelize(["hola", "PySpark"])
print(rdd_a.intersection(rdd_b).collect())

## subtract

Quita de un RDD los elementos que aparecen en otro.

**Tipo**: Transformación

In [ ]:
# Ejemplo 1: Elementos de rdd_a que no están en rdd_b
rdd_a = sc.parallelize([1, 2, 3, 4])
rdd_b = sc.parallelize([3, 4, 5, 6])
print(sorted(rdd_a.subtract(rdd_b).collect()))

In [ ]:
# Ejemplo 2: subtract no es simétrico, al revés da otro resultado
print(sorted(rdd_b.subtract(rdd_a).collect()))

In [ ]:
# Ejemplo 3: Restar todos los elementos deja un RDD vacío
rdd_a = sc.parallelize([1, 2])
rdd_b = sc.parallelize([1, 2])
print(rdd_a.subtract(rdd_b).collect())

In [ ]:
# Ejemplo 4: Restar palabras
rdd_a = sc.parallelize(["apple", "banana", "pear"])
rdd_b = sc.parallelize(["banana"])
print(sorted(rdd_a.subtract(rdd_b).collect()))

## cartesian

Devuelve todas las parejas posibles entre los elementos de dos RDD.

**Tipo**: Transformación

In [ ]:
# Ejemplo 1: Producto cartesiano de dos RDD
rdd_a = sc.parallelize([1, 2])
rdd_b = sc.parallelize(["a", "b"])
print(rdd_a.cartesian(rdd_b).collect())

In [ ]:
# Ejemplo 2: Con un RDD vacío el resultado es vacío
rdd_vacio = sc.parallelize([])
print(rdd_a.cartesian(rdd_vacio).collect())

In [ ]:
# Ejemplo 3: Producto cartesiano de un RDD consigo mismo
rdd = sc.parallelize([1, 2])
print(rdd.cartesian(rdd).collect())

## zip

Empareja dos RDD por posición. El primero con el primero, el segundo con el segundo, y así.

**Tipo**: Transformación

In [ ]:
# Ejemplo 1: Emparejar números con letras
rdd_a = sc.parallelize([1, 2, 3])
rdd_b = sc.parallelize(["a", "b", "c"])
print(rdd_a.zip(rdd_b).collect())

In [ ]:
# Ejemplo 2: Emparejar valores de distinto tipo
rdd_a = sc.parallelize([True, False])
rdd_b = sc.parallelize(["yes", "no"])
print(rdd_a.zip(rdd_b).collect())

In [ ]:
# Ejemplo 3: Emparejar flotantes con enteros
rdd_a = sc.parallelize([1.1, 2.2, 3.3])
rdd_b = sc.parallelize([1, 2, 3])
print(rdd_a.zip(rdd_b).collect())

## groupBy

Agrupa los elementos según el resultado de una función. Devuelve pares `(clave, iterable)`.

**Tipo**: Transformación

In [ ]:
# Ejemplo 1: Agrupar números por paridad
rdd = sc.parallelize([1, 2, 3, 4, 5, 6])
rdd_agrupado = rdd.groupBy(lambda x: x % 2)

print(rdd_agrupado.collect())

Cada grupo es un iterable y no una lista, así que hay que convertirlo para poder verlo.

In [ ]:
print({clave: list(grupo) for clave, grupo in rdd_agrupado.collect()})

In [ ]:
# Ejemplo 2: Agrupar palabras por su longitud
rdd_palabras = sc.parallelize(["one", "two", "three", "four"])

for clave, grupo in rdd_palabras.groupBy(len).collect():
    print(clave, ":", list(grupo))

In [ ]:
# Ejemplo 3: Agrupar números por su residuo módulo 3
rdd = sc.parallelize([1, 2, 3, 4, 5, 6, 7])
print({clave: list(grupo) for clave, grupo in rdd.groupBy(lambda x: x % 3).collect()})

## groupByKey

Agrupa un RDD de pares `(clave, valor)` por su clave. A diferencia de `groupBy`, no recibe ninguna función, porque la clave ya viene en el dato.

**Tipo**: Transformación

In [ ]:
# Ejemplo 1: Agrupar los valores de cada clave
rdd = sc.parallelize([("a", 1), ("b", 2), ("a", 3), ("b", 4)])
print({clave: list(grupo) for clave, grupo in rdd.groupByKey().collect()})

In [ ]:
# Ejemplo 2: Promedio de cada grupo
rdd = sc.parallelize([("A", 10), ("B", 20), ("A", 30), ("B", 40)])
print(rdd.groupByKey().mapValues(lambda valores: sum(valores) / len(valores)).collect())

In [ ]:
# Ejemplo 3: Agrupar ciudades por país
rdd = sc.parallelize([("MX", "Toluca"), ("AR", "Rosario"), ("MX", "Mérida")])

for pais, ciudades in rdd.groupByKey().collect():
    print(pais, ":", list(ciudades))

`groupByKey` manda todos los valores de una clave a un mismo nodo antes de operar. Cuando lo que sigue es una suma, un conteo o un máximo, `reduceByKey` llega al mismo resultado moviendo mucha menos información por la red.

## reduceByKey

Combina los valores de cada clave con una función. Va combinando dentro de cada nodo antes de mandar los datos por la red.

**Tipo**: Transformación

La función tiene que ser asociativa y conmutativa, porque Spark no garantiza en qué orden combina los valores.

In [ ]:
# Ejemplo 1: Sumar los valores de cada clave
rdd = sc.parallelize([("a", 1), ("b", 1), ("a", 2)])
print(sorted(rdd.reduceByKey(lambda a, b: a + b).collect()))

In [ ]:
# Ejemplo 2: Multiplicar los valores de cada clave
rdd = sc.parallelize([("x", 2), ("y", 3), ("x", 4), ("y", 3), ("z", 100)])
print(sorted(rdd.reduceByKey(lambda a, b: a * b).collect()))

In [ ]:
# Ejemplo 3: Máximo de cada clave
rdd = sc.parallelize([("cat", 10), ("dog", 30), ("cat", 50), ("dog", 130)])
print(sorted(rdd.reduceByKey(max).collect()))

## join

Combina dos RDD de pares por su clave. El resultado son pares `(clave, (valor_izquierdo, valor_derecho))`.

**Tipo**: Transformación

In [ ]:
# Ejemplo 1: Unir productos con sus precios
productos = sc.parallelize([("prod1", "TV"), ("prod2", "Laptop")])
precios = sc.parallelize([("prod1", 500), ("prod2", 1200)])

print(sorted(productos.join(precios).collect()))

In [ ]:
# Ejemplo 2: Las claves sin pareja se descartan
productos = sc.parallelize([("prod1", "TV"), ("prod3", "Tablet")])
precios = sc.parallelize([("prod1", 500), ("prod2", 1200)])

print(productos.join(precios).collect())

In [ ]:
# Ejemplo 3: leftOuterJoin conserva las claves del RDD izquierdo
print(sorted(productos.leftOuterJoin(precios).collect()))

# Acciones

Una acción dispara el cálculo y devuelve un resultado a Python o lo escribe en disco. Es lo que hace que se ejecuten las transformaciones pendientes.

## count

Cuenta los elementos del RDD.

**Tipo**: Acción

In [ ]:
# Ejemplo 1: Contar números
rdd = sc.parallelize([5, 6, 7, 8])
print(rdd.count())

In [ ]:
# Ejemplo 2: Contar cadenas de texto
rdd_texto = sc.parallelize(["spark", "hadoop", "flink"])
print(rdd_texto.count())

In [ ]:
# Ejemplo 3: Contar pares clave-valor
rdd_pares = sc.parallelize([("x", 1), ("y", 2), ("z", 3)])
print(rdd_pares.count())

## take

Devuelve los primeros n elementos del RDD.

**Tipo**: Acción

In [ ]:
# Ejemplo 1: Los primeros 3 elementos
rdd = sc.parallelize([10, 20, 30, 40, 50])
print(rdd.take(3))

In [ ]:
# Ejemplo 2: Las primeras 2 cadenas
rdd_texto = sc.parallelize(["blue", "green", "red", "purple"])
print(rdd_texto.take(2))

In [ ]:
# Ejemplo 3: El primer par clave-valor
rdd_pares = sc.parallelize([("p", 3), ("q", 4), ("r", 5)])
print(rdd_pares.take(1))

## first

Devuelve el primer elemento del RDD. Equivale a `take(1)`, pero sin la lista.

**Tipo**: Acción

In [ ]:
# Ejemplo 1: Primer número
rdd = sc.parallelize([15, 25, 35])
print(rdd.first())

In [ ]:
# Ejemplo 2: Primera cadena
rdd_texto = sc.parallelize(["uno", "dos", "tres"])
print(rdd_texto.first())

In [ ]:
# Ejemplo 3: Primer par clave-valor
rdd_pares = sc.parallelize([("c", 10), ("b", 20)])
print(rdd_pares.first())

## reduce

Combina los elementos de dos en dos hasta dejar uno solo.

**Tipo**: Acción

Como en `reduceByKey`, la función tiene que ser asociativa y conmutativa.

In [ ]:
# Ejemplo 1: Sumar todos los números
rdd = sc.parallelize([1, 2, 3, 4])
print(rdd.reduce(lambda a, b: a + b))

In [ ]:
# Ejemplo 2: Multiplicar todos los números
rdd = sc.parallelize([2, 3, 4])
print(rdd.reduce(lambda a, b: a * b))

In [ ]:
# Ejemplo 3: Encontrar el valor máximo
rdd = sc.parallelize([10, 25, 7, 89, 44])
print(rdd.reduce(max))

## countByKey

Cuenta cuántos elementos hay por cada clave.

**Tipo**: Acción

In [ ]:
# Ejemplo 1: Contar elementos por clave
rdd_pares = sc.parallelize([("a", 1), ("b", 2), ("a", 3), ("z", 122)])
print(rdd_pares.countByKey())

In [ ]:
# Ejemplo 2: Con una clave muy repetida
rdd_pares = sc.parallelize([("x", 1), ("y", 2), ("x", 3), ("x", 4)])
print(rdd_pares.countByKey())

In [ ]:
# Ejemplo 3: Con claves numéricas
rdd_pares = sc.parallelize([(1, "one"), (2, "two"), (1, "uno")])
print(rdd_pares.countByKey())

## saveAsTextFile

Guarda el RDD en disco. En lugar de un solo archivo genera una carpeta con un archivo por cada partición.

**Tipo**: Acción

Si la carpeta ya existe, Spark falla. Por eso los ejemplos la borran antes.

In [ ]:
import os
import shutil

# Ejemplo 1: Guardar un RDD de números
shutil.rmtree("numeros", ignore_errors=True)

rdd = sc.parallelize([5, 15, 25, 35])
rdd.saveAsTextFile("numeros")

print([f for f in sorted(os.listdir("numeros")) if not f.startswith(".")])

In [ ]:
# Ejemplo 2: Con 3 particiones se generan 3 archivos
shutil.rmtree("animales", ignore_errors=True)

rdd_texto = sc.parallelize(["cat", "dog", "mouse"], numSlices=3)
rdd_texto.saveAsTextFile("animales")

print([f for f in sorted(os.listdir("animales")) if not f.startswith(".")])

In [ ]:
# Ejemplo 3: Guardar pares clave-valor y volver a leerlos
shutil.rmtree("pares", ignore_errors=True)

rdd_pares = sc.parallelize([("key1", "val1"), ("key2", "val2")])
rdd_pares.saveAsTextFile("pares")

print(sc.textFile("pares").collect())

## takeSample

Devuelve una muestra aleatoria de elementos, con o sin reemplazo. Si fijas `seed`, la muestra siempre sale igual.

**Tipo**: Acción

In [ ]:
# Ejemplo 1: Muestra sin reemplazo
rdd = sc.parallelize(range(100))
print(rdd.takeSample(False, 10, seed=42))

In [ ]:
# Ejemplo 2: Con reemplazo, un elemento puede salir varias veces
rdd = sc.parallelize(range(10))
print(rdd.takeSample(True, 15, seed=42))

In [ ]:
# Ejemplo 3: Sin reemplazo no puede devolver más elementos de los que hay
rdd = sc.parallelize(range(10))
print(rdd.takeSample(False, 15, seed=42))

In [ ]:
# Ejemplo 4: La misma semilla devuelve siempre la misma muestra
rdd_texto = sc.parallelize(["apple", "banana", "cherry", "date", "fig"])

print(rdd_texto.takeSample(False, 3, seed=7))
print(rdd_texto.takeSample(False, 3, seed=7))

## top

Devuelve los n elementos mayores, según el orden natural.

**Tipo**: Acción

In [ ]:
# Ejemplo 1: Los 3 números más grandes
rdd = sc.parallelize([5, 10, 15, 20, 18])
print(rdd.top(3))

In [ ]:
# Ejemplo 2: Las 2 cadenas mayores en orden alfabético
rdd_texto = sc.parallelize(["spark", "hadoop", "flink", "mongo"])
print(rdd_texto.top(2))

In [ ]:
# Ejemplo 3: Con tuplas ordena por la tupla completa, empezando por la clave
rdd_pares = sc.parallelize([("a", 1), ("c", 2), ("b", 3)])
print(rdd_pares.top(2))

## max

Devuelve el elemento mayor.

**Tipo**: Acción

In [ ]:
# Ejemplo 1: Máximo de un RDD de números
rdd = sc.parallelize([10, 40, 20, 30])
print(rdd.max())

In [ ]:
# Ejemplo 2: Máximo de un RDD de cadenas, en orden alfabético
rdd_texto = sc.parallelize(["a", "z", "m", "n"])
print(rdd_texto.max())

In [ ]:
# Ejemplo 3: Máximo según el segundo elemento de cada tupla
rdd_tuplas = sc.parallelize([("a", 1), ("b", 5), ("c", 3)])
print(rdd_tuplas.max(key=lambda x: x[1]))

## min

Devuelve el elemento menor.

**Tipo**: Acción

In [ ]:
# Ejemplo 1: Mínimo de un RDD de números
rdd = sc.parallelize([10, 40, 20, 30])
print(rdd.min())

In [ ]:
# Ejemplo 2: Mínimo de un RDD de cadenas, en orden alfabético
rdd_texto = sc.parallelize(["a", "z", "m", "n"])
print(rdd_texto.min())

In [ ]:
# Ejemplo 3: Mínimo según el segundo elemento de cada tupla
rdd_tuplas = sc.parallelize([("a", 1), ("b", 5), ("c", 3)])
print(rdd_tuplas.min(key=lambda x: x[1]))

# Ejercicios finales

## Ejercicios con transformaciones

**1.** Crea un RDD de edades con `sc.parallelize([15, 18, 21, 14, 25])`. Descarta las menores de 18 y súmale 1 a las que quedan.

**2.** Con `sc.parallelize(["El veloz murciélago", "Hacia el bosque", "Hola mundo"])`, obtén un RDD con las palabras únicas.

**3.** Con `sc.parallelize([("A", 10), ("B", 20), ("A", 30), ("B", 40)])`, agrupa por clave con `groupByKey` y calcula el promedio de cada grupo.

**4.** Con `sc.parallelize([1, 2, 2, 3, 3, 3, 4])`, obtén los números sin repetir.

**5.** Combina `sc.parallelize([("prod1", "TV"), ("prod2", "Laptop")])` y `sc.parallelize([("prod1", 500), ("prod2", 1200)])` con `join`.

In [ ]:
# Problema 1


In [ ]:
# Problema 2


In [ ]:
# Problema 3


In [ ]:
# Problema 3 (variante)


In [ ]:
# Problema 4


In [ ]:
# Problema 5


## Ejercicios con acciones

**1.** Con `sc.parallelize([("TX1", 100), ("TX2", 150), ("TX3", 200)])`, cuenta las transacciones y muéstralas todas.

**2.** Con `sc.parallelize([1, 2, 3, 4, 5])`, calcula la suma usando `reduce`.

**3.** Con `sc.parallelize([65, 70, 68, 72, 75])`, obtén la temperatura mínima y la máxima.

**4.** Con `sc.parallelize([100, 101, 102, 103, 104, 105, 106])`, saca una muestra de 3 clientes sin reemplazo.

**5.** Con `sc.parallelize(["INFO Init", "ERROR Server", "INFO Shutdown", "ERROR Disk Full"])`, filtra los mensajes de ERROR y guárdalos con `saveAsTextFile`.

In [ ]:
# Problema 1


In [ ]:
# Problema 2


In [ ]:
# Problema 3


In [ ]:
# Problema 4


In [ ]:
# Problema 5


In [ ]:
# Liberar los recursos
spark.stop()